## Requirements
- Serverless v4

## A. Setup and Inspect Data

In [0]:
%pip install --upgrade databricks-langchain mlflow "unitycatalog-ai[databricks]==0.3.2" -qqq
%restart_python

### A1. Table and Functions in UC

In [0]:
catalog = "workspace"
schema = "bronze"
table_name = f"{catalog}.{schema}.sf_airbnb_listings"

# the functions below are written in the agent config YAML file
avg_function_name = f"{catalog}.{schema}.avg_neigh_price"
cnt_function_name = f"{catalog}.{schema}.cnt_by_room_type"


# check if pre-requisite functions exist (see 01- notebook)
def table_exists(table_name):
    catalog, schema, tbl = table_name.split('.')
    result = spark.sql(f"SHOW TABLES IN {catalog}.{schema} LIKE '{tbl}'")
    return result.count() > 0

def function_exists(function_name):
    result = spark.sql(f"SHOW USER FUNCTIONS IN {catalog}.{schema} LIKE '{function_name.split('.')[-1]}'")
    return result.count() > 0

print(table_exists(table_name))
print(function_exists(avg_function_name))
print(function_exists(cnt_function_name))

In [0]:
df = spark.read.table(table_name)
display(df.limit(5))

### A2. Tracing and Experiment
There are 2 approaches to store experiments:
- In a workspace location
- In a UC volume directory

In [0]:
import mlflow


mlflow.langchain.autolog()

# Location as user's homepage:
username = dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()
experiment_name_1 = f"/Workspace/Users/{username}/single_agents_demo1"
experiment_name_2 = f"/Workspace/Users/{username}/single_agents_demo2"

# Locatiopn as a UC volume:
artifact_path = f"dbfs:/Volumes/{catalog}/{schema}/agent_vol"

## C. Auto Tracing with MLflow Experiments

### C1. Tracing with Workspace Location
- Use `mlflow.set_experiment()`
- This approach is suitable for development and testing
- The underyling filesystem used is `dbfs:/databricks/mlflow-tracking/<experiment_id>`

In [0]:
mlflow.set_experiment(experiment_name_1)
artifact_location = mlflow.get_experiment_by_name(experiment_name_1).artifact_location
print(f"Artifact location: {artifact_location}")

### C3. Create the Agent

In [0]:
import os
from uuid import uuid4
from typing import Any, Dict, List

import yaml
import mlflow
from mlflow.pyfunc import ResponsesAgent
from mlflow.types.responses import ResponsesAgentRequest, ResponsesAgentResponse

from langchain.agents import create_agent
from databricks_langchain import ChatDatabricks, UCFunctionToolkit
from langgraph.checkpoint.memory import InMemorySaver


# load agent config from yaml file (in the same directory)
def _load_config(path: str = "agent-config.yaml") -> Dict[str, Any]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Config file not found at: {path}")
    with open(path, "r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f) or {}
    llm_endpoint = cfg.get("llm_endpoint")
    llm_temperature = float(cfg.get("llm_temperature"))
    system_prompt = cfg.get("system_prompt")
    function_names = cfg.get("function_names")

    return {
        "llm_endpoint": llm_endpoint,
        "llm_temperature": llm_temperature,
        "system_prompt": system_prompt,
        "function_names": function_names,
    }


# build LangChain agent with the config above:
# this is the same code as the smoke test above
def build_agent(
    llm_endpoint: str,
    system_prompt: str,
    llm_temperature: float = 0.1,
    function_names: list[str] = None,
):
    """
    Creates a UC-tool-calling ReAct agent with LangChain.
    Args:
        llm_endpoint (str): The endpoint of the LLM.
        system_prompt (str): The system prompt for the agent.
        llm_temperature (float): The temperature for the LLM.
        function_names (list[str]): The Unity Catalog fully-qualified names of the functions to be called. E.g., catalog.schema.function

    """

    # init the model with OpenAI standard I/O schemas
    llm = ChatDatabricks(endpoint=llm_endpoint, temperature=llm_temperature)

    # Use UCFunctionToolkit to integrate UC-registered tools
    if function_names:
        toolkit = UCFunctionToolkit(function_names=function_names)
        tools = toolkit.tools
    else:
        tools = []

    # Optional: use an in-memory saver to save the agent's state
    checkpointer = InMemorySaver()

    agent = create_agent(
        model=llm,
        tools=tools,
        system_prompt=system_prompt,
        checkpointer=checkpointer,
    )
    return agent

In [0]:
# `thread_id` is a unique identifier for a given conversation.
config = {"configurable": {"thread_id": "databricks-demo-1"}}

# init an agent
# function_names = [avg_function_name, cnt_function_name]
agent_config = _load_config()
agent = build_agent(
    llm_endpoint=agent_config["llm_endpoint"],
    system_prompt=agent_config["system_prompt"],
    llm_temperature=agent_config["llm_temperature"],
    function_names=agent_config["function_names"]
)

In [0]:
# quick smoke test
# note:
# with memory, the next time the same question is asked using the same parameters, the agent will rely on its memory and wont call the tools
# you need to re-init the agent above to reset its memory.
user_query = "Get the average price for Mission and tell me the number of properties there that have a shared room"

response = agent.invoke(
    {"messages":[{"role": "user", "content": user_query}]},
    config=config
)
print(response['messages'][-1].content)

### C3. The effect of memory
- When you reuse the same thread_id, the agent sees the full conversation history. The agent's reasoning: "I already answered this in our conversation, here's what I said before..."
- When you use a new thread ID, a new conversation is created, and the agent can call the tools again.
- Or when you re-init the agent with the old thread ID, the memory is erased.

In [0]:
# same thread ID as before, there is no tool calls from the traces
config = {"configurable": {"thread_id": "databricks-demo-1"}}
response = agent.invoke(
    {"messages":[{"role": "user", "content": user_query}]},
    config=config
)
print(response['messages'][-1].content)

In [0]:
# with a new thread ID, a new conversation is created, the agent calls tools again.
config = {"configurable": {"thread_id": "databricks-demo-2"}}
response = agent.invoke(
    {"messages":[{"role": "user", "content": user_query}]},
    config=config
)
print(response['messages'][-1].content)

## D. Manual Tracing

### D1. Simple Function Tracing

In [0]:
import mlflow
from mlflow.entities import SpanType


@mlflow.trace(
    span_type=SpanType.TOOL,
    name="Validate Input"
)
def validate_input(question: str, min_length: int = 5):
    """
    Checks if the user's question meets basic requirements.
    """
    if len(question) < min_length:
        return {
            "valid": False,
            "error": f"Question is too short (minimum {min_length} characters)"
        }
    if question.strip() == "":
        return {
            "valid": False,
            "error": "Question cannot be empty"
        }
    return {
        "valid": True,
        "cleaned_question": question.strip()
    }

@mlflow.trace(name="Process Question")
def process_question(user_input: str):
    """
    Processes and validate user input
    """
    validation_result = validate_input(user_input) # child span as TOOL
    if "cleaned_question" in validation_result.keys():
        cleaned = validation_result["cleaned_question"]
        return f"Processing: {cleaned}"
    else:
        error = validation_result["error"]
        return f"Error: {error}"
    


In [0]:
# Test with valid input
result = process_question("What is the average price of Mission?")
print(result)

In [0]:
# Test with invalid input -> throw errors and we can inspect it
result = process_question("What")
print(result)

### D2. Tracing LLM Calls with Custom Functions

In [0]:
import mlflow
from mlflow.entities import SpanType



# init an agent
function_names = [avg_function_name, cnt_function_name]
agent_config = _load_config()
agent = build_agent(
    llm_endpoint=agent_config["llm_endpoint"],
    system_prompt=agent_config["system_prompt"],
    llm_temperature=agent_config["llm_temperature"],
    function_names=function_names
)

# trace the agent
@mlflow.trace(
    name="Call LLM",
    span_type=SpanType.CHAT_MODEL
)
def call_llm(user_query: str):
    config = {"configurable": {"thread_id": "manual-llm-trace-1"}}
    response = agent.invoke(
        {"messages":[{"role": "user", "content": user_query}]},
        config=config
    )
    return response

@mlflow.trace(name="Process Question")
def process_question(user_input: str):
    """Main function that validates user input and calls LLM if valid input."""
    validation_result = validate_input(user_input)
    # call LLM to answer if question is valid
    if validation_result["valid"]:
        cleaned = validation_result["cleaned_question"]
        llm_response = call_llm(cleaned)
        return llm_response
    # return error if question is invalid
    else:
        error = validation_result["error"]
        return f"Error: {error}"

In [0]:
result = process_question("What is the average price of Mission?")

In [0]:
# with invalid result, the request is not sent to LLM as designed
result = process_question("Hi")